In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

print(sys.executable)  # should end in sec-10k-rag/.venv/bin/python

/home/nasim/.cache/uv/builds-v0/.tmpj7c85C/bin/python


In [3]:
from pathlib import Path

from sec_10k_rag.step_02_parsing.html_text import html_to_text
from sec_10k_rag.step_02_parsing.sections import split_into_items

In [4]:
html: bytes = Path("../data/raw/TSLA/2025/10k.html").read_bytes()
text: str = html_to_text(html)
print(text[:2000])

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
x | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
o | TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from _________ to _________
Commission File Number: 001-34756
Tesla, Inc.
(Exact name of registrant as specified in its charter)
Texas | 91-2197729
(State or other jurisdiction of
incorporation or organization) | (I.R.S. Employer
Identification No.)
1 Tesla Road
Austin, Texas
78725
(Address of principal executive offices) | (Zip Code)
(512) 516-8177
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class | Trading Symbol(s) | Name of each exchange on which registered
Common stock | TSLA | The Nasdaq Global Select Market
Securities registered pursuant to Section 12(g) o

In [5]:
print(text[2000:6000])

her the registrant is a large accelerated filer, an accelerated filer, a non-accelerated filer, a smaller reporting company, or an emerging growth company. See the definitions of “large accelerated filer,” “accelerated filer,” “smaller reporting company” and “emerging growth company” in Rule 12b-2 of the Exchange Act:
Large accelerated filer | x | Accelerated filer | o
Non-accelerated filer | o | Smaller reporting company | o
Emerging growth company | o
If an emerging growth company, indicate by check mark if the registrant has elected not to use the extended transition period for complying with any new or revised financial accounting standards provided pursuant to Section 13(a) of the Exchange Act. o
Indicate by check mark whether the Registrant has filed a report on and attestation to its management’s assessment of the effectiveness of its internal control over financial reporting under Section 404(b) of the Sarbanes-Oxley Act (15 U.S.C. 7262(b)) by the registered public accounting f

In [6]:
from sec_10k_rag.step_02_parsing.models import Heading, RawSection
from sec_10k_rag.step_02_parsing.sections import (
    body_start,
    find_headings,
    pick_body_headings,
)

In [7]:
print(f"{len(text):,} characters")

398,871 characters


In [8]:
headings: list[Heading] = find_headings(text)
print(len(headings), "heading-like lines")

46 heading-like lines


In [9]:
print(headings[:10])

[Heading(item='1', start=4865), Heading(item='1A', start=4882), Heading(item='1B', start=4904), Heading(item='1C', start=4939), Heading(item='2', start=4962), Heading(item='3', start=4981), Heading(item='4', start=5007), Heading(item='5', start=5048), Heading(item='6', start=5165), Heading(item='7', start=5184)]


In [10]:
for h in headings:
    line: str = text[h.start : text.find("\n", h.start)]
    print(f"{h.start:>8,}  {h.item:<3} {line}")

   4,865  1   Item 1.
   4,882  1A  Item 1A.
   4,904  1B  Item 1B.
   4,939  1C  Item 1C.
   4,962  2   Item 2.
   4,981  3   Item 3.
   5,007  4   Item 4.
   5,048  5   Item 5.
   5,165  6   Item 6.
   5,184  7   Item 7.
   5,278  7A  Item 7A.
   5,346  8   Item 8.
   5,398  9   Item 9.
   5,491  9A  Item 9A.
   5,524  9B  Item 9B.
   5,551  9C  Item 9C.
   5,638  10  Item 10.
   5,702  11  Item 11.
   5,734  12  Item 12.
   5,838  13  Item 13.
   5,921  14  Item 14.
   5,978  15  Item 15.
   6,030  16  Item 16.
   8,208  1   ITEM 1. BUSINESS
  53,458  1A  ITEM 1A. RISK FACTORS
 136,876  1B  ITEM 1B. UNRESOLVED STAFF COMMENTS
 136,917  1C  ITEM 1C. CYBERSECURITY
 142,644  2   ITEM 2. PROPERTIES
 143,872  3   ITEM 3. LEGAL PROCEEDINGS
 144,099  4   ITEM 4. MINE SAFETY DISCLOSURES
 144,155  5   ITEM 5. MARKET FOR REGISTRANT’S COMMON EQUITY, RELATED STOCKHOLDER MATTERS AND ISSUER PURCHASES OF EQUITY SECURITIES
 146,803  6   ITEM 6. [RESERVED]
 146,822  7   ITEM 7. MANAGEMENT’S DISCUSSIO

In [11]:
start: int = body_start(headings)
print("body starts at heading #", start, "->", headings[start])

body starts at heading # 23 -> Heading(item='1', start=8208)


In [12]:
chosen: list[Heading] = pick_body_headings(headings[start:])
print([h.item for h in chosen])

['1', '1A', '1B', '1C', '2', '3', '4', '5', '6', '7', '7A', '8', '9', '9A', '9B', '9C', '10', '11', '12', '13', '14', '15', '16']


In [13]:
sections: list[RawSection] = split_into_items(text)
for s in sections:
    print(f"{s.item:<3} {len(s.text):>9,}  {s.title:<45} {s.text[:50]!r}")

1      45,249  Business                                      'ITEM 1. BUSINESS\nOverview\nWe are focused on bringi'
1A     83,417  Risk Factors                                  'ITEM 1A. RISK FACTORS\nYou should carefully conside'
1B         40  Unresolved Staff Comments                     'ITEM 1B. UNRESOLVED STAFF COMMENTS\nNone.'
1C      5,726  Cybersecurity                                 'ITEM 1C. CYBERSECURITY\nCybersecurity Risk Manageme'
2       1,227  Properties                                    'ITEM 2. PROPERTIES\nWe are headquartered in Austin,'
3         226  Legal Proceedings                             'ITEM 3. LEGAL PROCEEDINGS\nFor a description of our'
4          55  Mine Safety Disclosures                       'ITEM 4. MINE SAFETY DISCLOSURES\nNot applicable.\nPA'
5       2,647  Market for Registrant's Common Equity         'ITEM 5. MARKET FOR REGISTRANT’S COMMON EQUITY, REL'
6          18  [Reserved]                                    'ITEM 6. [RESERVED]'
7      

In [14]:
from sec_10k_rag.step_02_parsing.sgml import extract_10k_html, split_documents

# the start of the 10-K's HTML

In [15]:
sub: str = Path("../data/raw/TSLA/2025/full_submission.txt").read_text(
    encoding="utf-8", errors="replace"
)
docs = split_documents(sub)

In [16]:
print(len(docs), "documents")
print([d.doc_type for d in docs])  # 10-K, EX-..., GRAPHIC, XML, ...
print(extract_10k_html(sub)[:500])

118 documents
['10-K', 'EX-4.17', 'EX-10.9', 'EX-10.10', 'EX-21.1', 'EX-23.1', 'EX-31.1', 'EX-31.2', 'EX-32.1', 'EX-101.SCH', 'EX-101.CAL', 'EX-101.DEF', 'EX-101.LAB', 'EX-101.PRE', 'GRAPHIC', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'JSON', 'ZIP', 'XML']


<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva P

In [17]:
def test_missing_item_is_skipped(fixtures_dir: Path) -> None:
    file_path = fixtures_dir / "missing_items_in_filing.txt"
    test_data = file_path.read_text(encoding="utf-8")
    sections = split_into_items(test_data)
    expected_items = ["1", "7"]
    assert [section.item for section in sections] == expected_items
    assert sections[0].text.startswith("Item 1. Business\nWe design")
    assert sections[-1].text.endswith("Total revenues grew 3% in 2025.")

In [18]:
# test_missing_item_is_skipped(Path(__file__).parent / "..")
# from pathlib import Path
# print(Path.cwd().parent / "fixtures/missing_items_in_filing.txt")
test_missing_item_is_skipped(Path.cwd().parent / "tests/fixtures")

In [19]:
import json
from collections.abc import Iterator
from pathlib import Path
from typing import Any

type Json = dict[str, Json] | list[Json] | str | int | float | bool | None
type Scalar = str | int | float | bool | None
type Row = dict[str, Scalar]

LEVEL_PREFIX = "__col"

# type Json = dict[str, Json] | list[Json] | str | int | float | bool | None
# type Row = dict[str, Json]
# type Position = dict[str, str | int]
ROOT_DIR = Path.cwd().parent
DATA_RAW_DIR = Path("data/raw")

In [20]:

def load_company_facts(
    source_dir: Path, flattening_map: list[tuple[str, dict[str, Any]]]
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    for facts_path in sorted(source_dir.glob("*/companyfacts.json")):
        ticker: str = facts_path.parent.name  # "TSLA", taken from the folder name
        company_facts: dict[str, Any] = json.loads(facts_path.read_text(encoding="utf-8"))
        print(ticker, company_facts["entityName"])
        print(company_facts["facts"]["dei"]["EntityCommonStockSharesOutstanding"]["label"])
        # flatten_data(company_facts, flattening_map)

    return rows

In [21]:
# print(ROOT_DIR / DATA_RAW_DIR)
company_ticker = "TSLA"
facts_file_name = "__companyfacts.json"
full_file_name = ROOT_DIR / DATA_RAW_DIR / company_ticker / facts_file_name
print(full_file_name)
# print(full_file_name.exists())
# print(full_file_name.is_file())
company_facts: dict[str, Any] = json.loads(full_file_name.read_text(encoding="utf-8"))
print(company_facts.keys())
facts = company_facts["facts"]
print(facts.keys())
dei = facts["dei"]
print(dei.keys())
content = dei["EntityCommonStockSharesOutstanding"]
print(content.keys())
unit = content["units"]
print(unit.keys())
records = unit["shares"]
print(len(records))
for item in records[:2]:
    # print(item.keys())
    for key in item:
        print(f"{key:>10}: {item[key]}")

/home/nasim/Desktop/work/rag/sec-10k-rag/data/raw/TSLA/__companyfacts.json
dict_keys(['cik', 'entityName', 'facts'])
dict_keys(['dei', 'us-gaap', 'ffd'])
dict_keys(['EntityCommonStockSharesOutstanding', 'EntityPublicFloat'])
dict_keys(['label', 'description', 'units'])
dict_keys(['shares'])
67
       end: 2011-07-31
       val: 104018274
      accn: 0001193125-11-221497
        fy: 2011
       sub: {'final': 'GOOD', 'next': 'proceed'}
      sub2: {'final': 'NO_NEED', 'next': 'none'}
        fp: Q2
      form: 10-Q
     filed: 2011-08-12
     frame: CY2011Q2I
       end: 2011-10-31
       val: 104298634
      accn: 0001193125-11-308489
        fy: 2011
        fp: Q3
      form: 10-Q
     filed: 2011-11-14
     frame: CY2011Q3I


In [22]:
# ROOT_DIR = Path.cwd().parent
map_path = "src/sec_10k_rag/step_03_flattening/flattening_map.json"
config = json.loads((ROOT_DIR / map_path).read_text(encoding="utf-8"))

# mapping: dict[str, dict[str, Any]] = config["map_test"]


def sort_label(item: tuple[str, dict[str, Any]]) -> tuple[int, str]:
    name, spec = item
    depth: int = spec["from"].count(".")
    return (depth, name)  # sort by depth first, then by name


# ordered_map: list[tuple[str, dict[str, Any]]] = sorted(
#     mapping.items(),
#     key=sort_label
#     # key=lambda item:(item[1]["from"].count("."), item[0])
# )


# for index, (fld, spec) in enumerate(ordered_map):
#     node_path = spec.get("from", "").split(".")
#     src_fld = spec.get("key", "")
#     allowed_values= spec.get("allowed_values",[])
#     # node = get_node(company_records, current_index, node_path, company_facts)




In [23]:
def leaf_path_lambda(mapping: dict[str, dict[str, Json]]) -> list[str]:
    """The deepest "from" path in the map, split into segments."""
    return max((str(spec["from"]) for spec in mapping.values()), key=lambda p: p.count(".")).split(
        "."
    )

In [24]:
def leaf_path_lambda2(mapping: dict[str, dict[str, Json]]) -> list[str]:
    """The deepest "from" path in the map, split into segments."""
    return max((str(spec["from"]) for spec in mapping.values()), key=lambda p: p.count(".")).split(
        "."
    )

In [25]:
def leaf_path(mapping: dict[str, dict[str, Json]]) -> list[str]:
    """The deepest "from" path in the map, split into segments."""
    return max([str(spec["from"]).split(".") for spec in mapping.values()], key=len)

In [26]:
print([max(["testing", "no", "whats is up"], key=len)])

['whats is up']


In [27]:
def fold(node: Json, prefix: str = "") -> Row:
    """Turn everything inside one row into columns named by path: sub_final, tags_0, ..."""
    if isinstance(node, dict):
        pairs = [(f"{prefix}{k}_", child) for k, child in node.items()]
        return {col: v for p, child in pairs for col, v in fold(child, p).items()}
    if isinstance(node, list):
        pairs = [(f"{prefix}{i}_", child) for i, child in enumerate(node)]
        return {col: v for p, child in pairs for col, v in fold(child, p).items()}
    return {prefix[:-1] or "value": node}

In [28]:
def flatten(node: Json, row: Row, depth: int = 1) -> Iterator[Row]:
    """Dict keys above the first list become __col1, __col2, ...; each list item is a row."""
    if isinstance(node, list):
        for item in node:
            yield {**row, **fold(item)}
        return
    if not isinstance(node, dict):
        return
    fields = {k: v for k, v in node.items() if not isinstance(v, dict | list)}
    nested = {k: v for k, v in node.items() if isinstance(v, dict | list)}
    if not nested:
        yield {**row, **fields}
        return
    for key, child in nested.items():
        yield from flatten(child, {**row, **fields, f"{LEVEL_PREFIX}{depth}": key}, depth + 1)

In [29]:
def rename(row: Row, rename_map: dict[str, str], drop: list[str]) -> Row:
    """Rename colN to real names; drop unmapped colN and listed columns."""
    return {
        rename_map.get(k, k): v
        for k, v in row.items()
        if k not in drop and (k in rename_map or not k.startswith(LEVEL_PREFIX))
    }


def keep(row: Row, allowed: dict[str, list[str]]) -> bool:
    """True if every filtered column's value is in its allowed list."""
    return all(row.get(col) in values for col, values in allowed.items())